# Exam 2: Extra prep (solutions)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/engr1050-fall2026/blob/main/NewMaterial/Lecture19/exam2_extraPrep_solutions.ipynb)


## Representative cheat sheet

A cheat sheet like this one will be provided with the exam, as in Exam 1.

<table>
<tr>
<td valign="top">
<pre>
# numpy
import numpy as np
v = np.array([1, 2, 3])   # a vector
v[0]                      # 1 (first entry)
v[-1]                     # 3 (last entry)
v + v                     # [2, 4, 6] entry by entry
2 * v                     # [2, 4, 6]
M = np.array([[1, 2],
              [3, 4]])    # 2x2 matrix (list of rows)
M[1, 0]                   # 3 (row 1, column 0)
M[0, :]                   # [1, 2] (first row)
M[:, 1]                   # [2, 4] (second column)
np.zeros(3)               # [0., 0., 0.]
np.zeros((3, 3))          # 3x3 matrix of zeros

# Lists and loops
vals = [0.5]
vals.append(1.5)          # [0.5, 1.5]
for n in range(3):        # n = 0, 1, 2
    print(n)
x = 0
while x &lt; 3:              # repeats while True
    x += 1                # same as x = x + 1

# Conditionals and logic
if x &lt; 5:
    print("small")
elif x == 5:
    print("five")
else:
    print("big")
x == 5    x != 5    x &lt;= 5    x &gt;= 5
a and b   # True if both are True
a or b    # True if either is True
abs(-3)                   # 3
</pre>
</td>
<td valign="top">
<pre>
# Functions
def add(a, b):
    return a + b
add(3, 5)                 # 8

# Classes
class Dog:
    def __init__(self, name):
        self.name = name
    def bark(self):
        return "Woof! I'm " + self.name
myDog = Dog("Fido")       # runs __init__
myDog.name                # "Fido"
myDog.bark()              # "Woof! I'm Fido"

# Inheritance
class Puppy(Dog):         # gets all of Dog
    def bark(self):       # replaces Dog's bark
        return "Yip!"

class ShowDog(Dog):
    def __init__(self, name, prize):
        super().__init__(name)  # Dog's __init__
        self.prize = prize      # plus a new one
ShowDog("Rex", "gold")

# Pico
from machine import Pin, ADC
import time
led = Pin(5, Pin.OUT)     # output pin
led.on()                  # led.off() turns off
btn = Pin(7, Pin.IN, Pin.PULL_DOWN)  # input
btn.value()               # 1 if pressed, else 0
pot = ADC(Pin(26))        # analog input pin
pot.read_u16()            # reading, 0 to 65535
time.sleep(0.1)           # wait 0.1 seconds
</pre>
</td>
</tr>
</table>


Exam 2 covers the material since Exam 1: numpy, ODEs and initial value problems, solving ODEs numerically with Explicit Euler and our `ODE_solver` class, proportional control of the helicopter, and the Pico labs with the potentiometer, buttons, and LEDs. Like Exam 1, it is closed book and closed notes. Most of the exam draws on problems from the in-class exercises, the homeworks, and the review, so the problems below are a good check that you are comfortable with them. Try each one yourself before looking anything up.

This notebook contains worked solutions. Try each problem in the [practice version](https://colab.research.google.com/github/natrask/engr1050-fall2026/blob/main/NewMaterial/Lecture19/exam2_extraPrep.ipynb) before looking here. Many problems have more than one correct answer, so a solution that differs from the one below can still be right.

> **A note on inputs.** On the exam, unless a question says otherwise, you do not need to clean up or check the input. If a question says an input is a list of numbers, or a string, you may assume it really is a list of numbers or a string. For example, you do not need to check whether a list of numbers accidentally contains strings.

> **Use of built-in Python functionality.** The purpose of this unit of the course is to assess whether you understand how to use the Python basics (if/for/while/classes/functions/etc.) to perform scientific computing. We do not want you to memorize built-in Python functions - we want you to use basic numpy and the `ODE_solver` class we designed together in order to complete tasks.

## Basic numpy

1. **A reminder about numpy vectors.** Import numpy once at the top of your code, then build a vector by wrapping `np.array()` around a list:

   ```python
   import numpy as np

   v = np.array([1, 2, 3])   # a vector
   v[0]                      # first entry: 1
   v[-1]                     # last entry: 3

   w = np.array([4, 5, 6])
   v + w                     # adds entry by entry: [5, 7, 9]
   2 * v                     # multiplies every entry: [2, 4, 6]
   ```

   Construct the vectors $x = [1, 5]$ and $y = [2, 3]$ as numpy arrays and print out their sum.


In [1]:
import numpy as np

x = np.array([1, 5])
y = np.array([2, 3])
print(x + y)    # adds entry by entry


[3 8]


2. **A reminder about numpy matrices.** A matrix is a list of rows, so you build one by wrapping `np.array()` around a list of lists. Inside the brackets, the first number picks the row and the second picks the column, both counting from 0:

   ```python
   M = np.array([[1, 2],
                 [3, 4]])    # a 2x2 matrix; [1, 2] is the first row
   M[1, 0]                   # second row, first column: 3
   M[0, :]                   # the whole first row: [1, 2]
   M[:, 1]                   # the whole second column: [2, 4]
   ```

   Consider the following linear system for the unknowns $x = [x_1, x_2, x_3]$:

   $$
   \begin{aligned}
   2x_1 + x_2 - x_3 &= 1 \\
   x_1 + 3x_2 + 2x_3 &= 13 \\
   -x_1 + 4x_3 &= 11
   \end{aligned}
   $$

   Initialize the matrix `A` and the vector `b` so that this system reads $Ax = b$.


In [2]:
import numpy as np

# Each row of A holds the coefficients of one equation, in the order x1, x2, x3.
# The third equation has no x2, so its row gets a 0 in that spot.
A = np.array([[ 2, 1, -1],
              [ 1, 3,  2],
              [-1, 0,  4]])
b = np.array([1, 13, 11])
print(A)
print(b)


[[ 2  1 -1]
 [ 1  3  2]
 [-1  0  4]]
[ 1 13 11]


**Check.** The solution of this system is $x = [1, 2, 3]$, and it satisfies all three equations: $2(1) + 2 - 3 = 1$, $1 + 3(2) + 2(3) = 13$, and $-1 + 4(3) = 11$. A common mistake is to leave out the 0 for the missing $x_2$ in the third equation, which makes that row too short.

## Fundamentals of ODEs

Consider the following ODE for $y(t)$:

$$
y'' + 3y' + 2y = 4
$$

These questions are pencil and paper, the way they will be on the exam. The worked answers are in the text cell below each question.

3. What is the order of this ODE? How many initial conditions does it need to have a unique solution?


**Answer.** The highest derivative is $y''$, so the ODE is **second order**. A second-order ODE needs **two** initial conditions to have a unique solution: one for $y(0)$ and one for $y'(0)$.

4. Rewrite this ODE as a system of first-order ODEs.


**Answer.** Introduce a new unknown $v = y'$. Then $v' = y''$, and solving the ODE for $y''$ gives $y'' = 4 - 3y' - 2y$. The system is

$$
\begin{aligned}
y' &= v \\
v' &= 4 - 3v - 2y
\end{aligned}
$$

In the `ODE_solver` class, with `y[0]` holding $y$ and `y[1]` holding $v$, this would be `f[0] = y[1]` and `f[1] = 4 - 3*y[1] - 2*y[0]`.

5. What is the steady state solution of this ODE?


**Answer.** At steady state nothing changes, so every derivative is zero: $y' = 0$ and $y'' = 0$. The ODE becomes $0 + 0 + 2y = 4$, so the steady state is $y = 2$.

The system from question 4 gives the same answer. Setting $y' = 0$ and $v' = 0$, the first equation gives $v = 0$, and the second gives $0 = 4 - 2y$, so $y = 2$.

6. Confirm that the analytic solution

   $$
   y(t) = 2 - 4e^{-t} + 2e^{-2t}
   $$

   satisfies the ODE. What are the initial conditions that would make this a solution to an initial value problem? Make sure you give the correct number of initial conditions.


**Answer.** Differentiate twice:

$$
\begin{aligned}
y &= 2 - 4e^{-t} + 2e^{-2t} \\
y' &= 4e^{-t} - 4e^{-2t} \\
y'' &= -4e^{-t} + 8e^{-2t}
\end{aligned}
$$

Substitute into the left side and collect like terms:

$$
\begin{aligned}
y'' + 3y' + 2y &= \left(-4e^{-t} + 8e^{-2t}\right) + \left(12e^{-t} - 12e^{-2t}\right) + \left(4 - 8e^{-t} + 4e^{-2t}\right) \\
&= 4 + (-4 + 12 - 8)\,e^{-t} + (8 - 12 + 4)\,e^{-2t} \\
&= 4
\end{aligned}
$$

This is the right side of the ODE, so $y(t)$ is a solution.

The ODE is second order, so it needs two initial conditions. Setting $t = 0$, where $e^{0} = 1$:

$$
y(0) = 2 - 4 + 2 = 0, \qquad y'(0) = 4 - 4 = 0
$$

So $y(t)$ solves the initial value problem with $y(0) = 0$ and $y'(0) = 0$. As a check, $y(t) \to 2$ as $t \to \infty$, which matches the steady state from question 5.

## Implementing Explicit Euler

Explicit Euler solves the initial value problem $y' = f(t, y)$ with $y(t_0) = y_0$ by taking steps of size $h$. Each step uses the current values to compute the next ones:

$$
t_{n+1} = t_n + h, \qquad y_{n+1} = y_n + h \, f(t_n, y_n)
$$

In pseudocode:

```text
given: the right-hand side f(t, y), the initial condition t0 and y0,
       the step size h, and the number of steps N

set t = t0 and y = y0, and record them
repeat N times:
    y = y + h * f(t, y)      # update y first, using the current t
    t = t + h
    record t and y
```

7. Use a `for` loop to implement Explicit Euler for the ODE

   $$
   y' = -2y, \qquad y(0) = 1
   $$

   taking 10 timesteps of size $h = 0.1$. Do not use classes or any advanced functionality. Your code should use a single `for` loop and output a list or numpy array of the time values and a list or numpy array of the solution values.


In [3]:
h = 0.1        # step size
N = 10         # number of steps

t = 0.0        # initial time
y = 1.0        # initial condition y(0) = 1
tValues = [t]
yValues = [y]

for n in range(N):
    y = y + h * (-2 * y)    # f(t, y) = -2y; update y first, using the current t
    t = t + h
    tValues.append(t)
    yValues.append(y)

# Print the two lists side by side
for n in range(N + 1):
    print(f"t = {tValues[n]:.1f}   y = {yValues[n]:.6f}")


t = 0.0   y = 1.000000
t = 0.1   y = 0.800000
t = 0.2   y = 0.640000
t = 0.3   y = 0.512000
t = 0.4   y = 0.409600
t = 0.5   y = 0.327680
t = 0.6   y = 0.262144
t = 0.7   y = 0.209715
t = 0.8   y = 0.167772
t = 0.9   y = 0.134218
t = 1.0   y = 0.107374


**Explanation.** Each step multiplies $y$ by $1 - 2h = 0.8$, so after 10 steps $y = 0.8^{10} \approx 0.107$. The exact solution is $y(t) = e^{-2t}$, so $y(1) = e^{-2} \approx 0.135$. The difference is the error of Explicit Euler with $h = 0.1$, and a smaller step size shrinks it.

The last loop only prints the values neatly. Printing the lists directly is also fine, but the times then show rounding noise such as `0.30000000000000004`.

## Understanding the `ODE_solver` class

Here is the `ODE_solver` class that we built in class and used in the homework. Run this cell before starting the questions below.

In [4]:
import numpy as np

class ODE_solver:
    def __init__(self, t0, tf, y0, N):
        self.y0 = y0    # initial condition (should be a numpy array)
        self.t0 = t0    # initial time
        self.tf = tf    # final time
        self.N = N      # number of steps
        self.h = (tf - t0) / float(N) # step size, calculated from inputs

        # We need to check what size y0 is and store that information to size the solution array
        self.dim = len(y0) #assumes that y0 is a list or array

        # Create a place to store the solution
        self.t_values = np.array([t0 + i*self.h for i in range(N+1)])  # time values
        self.solution = np.zeros((N+1, self.dim))  # solution array
        self.solution[0, :] = y0  # set initial condition

        print(f"{self.__class__.__name__} initialized!")  # success
        
    def f(self,t,y):
        # Define the right-hand side of the ODE here, assuming a scalar t and a numpy array y
        # Example: dy/dt = -2y (for scalar y),
        # this function should return a numpy array of the same shape as y
        f = np.zeros(self.dim) # use self.dim to match shape of y
        f[0] = -2 * y[0]  # Example for the first component
        return f
    
    def solve(self):
        for n in range(self.N):
            t_n = self.t_values[n]
            y_n = self.solution[n, :]
            f_n = self.f(t_n, y_n)
            self.solution[n+1, :] = y_n + self.h * f_n
        return self.solution


8. Define a class that inherits from `ODE_solver` and specializes it to the skydiver ODE

   $$
   y' = g - \frac{k}{m} y^2, \qquad y(0) = 0
   $$

   with $g = 9$, $k = 1$, and $m = 1$.


In [5]:
class Skydiver(ODE_solver):
    # Inherit everything from ODE_solver and replace only f
    def f(self, t, y):
        g = 9
        k = 1
        m = 1
        f = np.zeros(self.dim)
        f[0] = g - (k/m) * y[0]**2
        return f


**Explanation.** The child class only needs a new `f`. It gets `__init__` and `solve` from `ODE_solver` unchanged. The skydiver ODE has a single unknown, `y[0]`, so `f` has a single entry.

9. Read the `ODE_solver` initializer (the `__init__` function) and write down in words what inputs should be provided to the class.


**Answer.** The initializer takes four inputs, in this order:

- `t0`, the initial time.
- `tf`, the final time.
- `y0`, the initial condition. It must be a numpy array (or a list) with one entry for each unknown, even when there is only one, because the initializer calls `len(y0)`. For a single ODE, pass `np.array([0.0])`, not `0.0`.
- `N`, the number of timesteps.

The step size is not an input. The initializer computes it from the others as `h = (tf - t0) / N`.

10. For the skydiver ODE

    $$
    y' = g - \frac{k}{m} y^2, \qquad y(0) = 0
    $$

    with $g = 9$, $k = 1$, and $m = 1$, write three initializations of your class from question 8 that solve the ODE from time $t = 0$ to $t = 1$ with 10, 20, and 40 timesteps.


In [6]:
y0 = np.array([0.0])    # y(0) = 0, as a one-entry array

skydiver10 = Skydiver(0, 1, y0, 10)    # h = 0.1
skydiver20 = Skydiver(0, 1, y0, 20)    # h = 0.05
skydiver40 = Skydiver(0, 1, y0, 40)    # h = 0.025


Skydiver initialized!
Skydiver initialized!
Skydiver initialized!


**Explanation.** All three solve the same problem from $t = 0$ to $t = 1$. Only `N` changes, so the step sizes are $h = 0.1$, $0.05$, and $0.025$. The initial condition is a one-entry numpy array, as question 9 explains.

11. Two people each provide you with a solution to the skydiver ODE from question 8, with $g = 9$, $k = 1$, and $m = 1$. For each one, use the steady state solution to confirm whether you trust their code output.


**a.** The first solution:

<div align="center">
    <img src="https://raw.githubusercontent.com/natrask/engr1050-fall2026/main/NewMaterial/_shared/Images/exam2_skydiver_solution_a.png" alt="First numerical solution of the skydiver ODE" width="450">
</div>


**Answer.** First find the steady state. At steady state $y' = 0$, so $0 = 9 - y^2$, which gives $y = 3$. (The skydiver starts from rest, $y(0) = 0$, and speeds up, so $y$ stays positive and we take the positive root.)

The first solution levels off at 3, which matches the steady state, so we trust it.

**b.** The second solution:

<div align="center">
    <img src="https://raw.githubusercontent.com/natrask/engr1050-fall2026/main/NewMaterial/_shared/Images/exam2_skydiver_solution_b.png" alt="Second numerical solution of the skydiver ODE" width="450">
</div>


**Answer.** The second solution levels off at 6, not 3, so we do not trust it. At $y = 6$ the ODE gives $y' = 9 - 36 = -27$, not 0, so 6 cannot be a steady state of this ODE. The code that produced it must be solving a different equation.

12. Given the following code, what is the ODE that it solves?

    ```python
    class childclassODE(ODE_solver):
        def __init__(self, t0, tf, y0, N, newparameter):
            super().__init__(t0, tf, y0, N)
            self.newparameter = float(newparameter)
        def f(self, t, y):
            f = np.zeros(self.dim)
            f[0] = 1 - self.newparameter * y[0]**2
            return f
    ```


**Answer.** The class replaces `f` with `f[0] = 1 - self.newparameter * y[0]**2`, so it solves

$$
y' = 1 - c\,y^2
$$

where $c$ is the value of `newparameter` given when the object is created. The initializer takes the same `t0`, `tf`, `y0`, and `N` as `ODE_solver` and passes them on with `super().__init__`, so the initial condition is whatever `y0` is. Its one new job is to store the extra input as `self.newparameter`. This is the skydiver ODE with $g = 1$ and $k/m = c$.

## Proportional control

13. For the driven pendulum equation modeling the helicopter, we derived the following ODE, linearized for small angles:

    $$
    \theta'' = -b\theta' - K\theta + K_p (\theta_{\text{target}} - \theta)
    $$


**a.** What is the steady state value of this ODE?


**Answer.** At steady state the helicopter is not moving, so $\theta' = 0$ and $\theta'' = 0$, and $\theta = \theta_{ss}$. Substituting:

$$
0 = -b \cdot 0 - K\theta_{ss} + K_p (\theta_{\text{target}} - \theta_{ss})
$$

Move the $\theta_{ss}$ terms to one side: $K\theta_{ss} + K_p\theta_{ss} = K_p\theta_{\text{target}}$. So

$$
\theta_{ss} = \frac{K_p}{K + K_p}\,\theta_{\text{target}}
$$

The damping $b$ drops out. It changes how the helicopter gets to its steady state, but not where it ends up.

**b.** If $\theta_{\text{target}}$ is chosen as 90 degrees, how much power $K_p$ must be provided to reach the desired steady state of 90 degrees? Use your steady state value to reason through this.


**Answer.** With $\theta_{\text{target}} = 90$ degrees, reaching $\theta_{ss} = 90$ degrees requires

$$
\frac{K_p}{K + K_p} \cdot 90 = 90, \qquad \text{that is,} \qquad \frac{K_p}{K + K_p} = 1
$$

Gravity makes $K > 0$, so the denominator $K + K_p$ is always larger than $K_p$, and the fraction is always less than 1. A larger $K_p$ brings $\theta_{ss}$ closer to 90 degrees, but no finite $K_p$ reaches it: it would take infinite power. Proportional control always stops short of its target. This is the steady-state error we saw in class and in HW4.

**c.** Propose a different $\theta_{\text{target}}$ that will allow you to achieve a steady state of 90 degrees if $K_p = 1$.


**Answer.** Aim past 90 degrees. With $K_p = 1$, the steady state is

$$
\theta_{ss} = \frac{1}{K + 1}\,\theta_{\text{target}}
$$

Setting $\theta_{ss} = 90$ and solving for the target gives

$$
\theta_{\text{target}} = 90\,(K + 1) \text{ degrees}
$$

For example, if $K = 2$, set $\theta_{\text{target}} = 270$ degrees. The controller still stops short of its target, but the target is chosen so that it stops at exactly 90 degrees.

## The Pico labs

The next problems check that you have been following the Pico labs. Their code runs on the Pico through Thonny, not in Colab, so do not try to run it here. The answer is in the cell below each question.

14. The following program reads the potentiometer on the pendulum and prints the raw reading:

    ```python
    from machine import ADC, Pin
    import time

    potentiometer = ADC(Pin(26))

    while True:
        pot_value = potentiometer.read_u16()
        print(pot_value)
        time.sleep(0.1)
    ```

    When the pendulum hangs at 0 degrees, `pot_value` reads 34000. When the pendulum is at 90 degrees, it reads 9200. What modifications need to be made to this program so that it prints the angle of the pendulum in degrees? Make them in the cell below.


In [ ]:
from machine import ADC, Pin
import time

potentiometer = ADC(Pin(26))

# Calibration readings
theta0 = 34000     # pot_value when the pendulum hangs at 0 degrees
theta90 = 9200     # pot_value when the pendulum is at 90 degrees

while True:
    pot_value = potentiometer.read_u16()
    # The straight line through (34000, 0 degrees) and (9200, 90 degrees)
    theta_value = 90.0 * (pot_value - theta0) / (theta90 - theta0)
    print(theta_value)
    time.sleep(0.1)


**Explanation.** There are three changes: store the two calibration readings, add a line that converts `pot_value` to `theta_value`, and print `theta_value` instead of `pot_value`. The conversion is the straight line through the two calibration points. Check the endpoints: when `pot_value` is 34000 the top of the fraction is 0, so `theta_value` is 0, and when `pot_value` is 9200 the top and bottom are equal, so `theta_value` is 90.

The reading goes *down* as the angle goes up, which is why `theta90 - theta0` is negative. The formula handles that without any special case. Writing the numbers in directly, `theta_value = 90.0 * (pot_value - 34000) / (9200 - 34000)`, is equally correct.

15. Here are the `LEDClass` and `ButtonClass` classes from the labs:

    ```python
    from machine import Pin
    import time

    class LEDClass:
        def __init__(self, pin_number, initial_state=False):
            self.led = Pin(pin_number, Pin.OUT)
            self.state = initial_state
            if self.state:
                self.led.on()
            else:
                self.led.off()

        def toggle(self):
            if self.state:
                self.led.off()
                self.state = False
            else:
                self.led.on()
                self.state = True

    class ButtonClass:
        def __init__(self, pin_number):
            self.button = Pin(pin_number, Pin.IN, Pin.PULL_DOWN)
            self.oldbuttonstate = self.button.value()

        def catchPress(self):
            newbuttonstate = self.button.value()
            if self.oldbuttonstate == 0 and newbuttonstate == 1:
                self.oldbuttonstate = newbuttonstate
                return True
            self.oldbuttonstate = newbuttonstate
            time.sleep(0.01)
            return False
    ```

    Assume the Pico's `LED1` is wired to `GP16` and its button `K1` is wired to `GP14`.


**a.** Create an instance of each class, `led1` and `button1`, so that they are linked to `LED1` and `K1`.


In [ ]:
led1 = LEDClass(16)        # LED1 is wired to GP16
button1 = ButtonClass(14)  # K1 is wired to GP14


**Explanation.** Each class takes the `GP` pin number as its input. `initial_state` is left at its default of `False`, so `LED1` starts off.

**b.** Consider the following function:

```python
def mystery(x, y):
    while True:
        if x.catchPress():
            y.toggle()
```

Explain in words what happens when `mystery(button1, led1)` is executed.


**Answer.** `mystery(button1, led1)` turns `K1` into an on/off switch for `LED1`. Inside the function, `x` is `button1` and `y` is `led1`.

The `while True` loop checks the button over and over, forever. `catchPress()` returns `True` only at the moment the button goes from not pressed (0) to pressed (1), so each press counts once, no matter how long the button is held down. Each press calls `toggle()`, which flips the LED. `LED1` starts off, so the first press turns it on, the second turns it off, the third turns it on again, and so on. The function never returns; it keeps running until you press Stop in Thonny.

16. In the helicopter user interface lab, one of the design requirements was:

    > Set one LED to turn off/on depending on whether the pendulum is within 5 degrees of the target angle.

    The on-target LED was created from the lab's `LEDClass`:

    ```python
    class LEDClass:
        def __init__(self, pin_number, initial_state=False):
            """
            Initialize LED class
            pin_number: GPIO pin number for the LED
            initial_state: True for on, False for off (default: False)
            """
            self.led = Pin(pin_number, Pin.OUT)
            self.state = initial_state
            
            # Set initial state
            if self.state:
                self.led.on()
            else:
                self.led.off()
        
        def toggle(self):
            """Toggle the LED state (on->off or off->on)"""
            if self.state:
                self.led.off()
                self.state = False
            else:
                self.led.on()
                self.state = True
    ```

    Assume you already have the calibrated angle of the pendulum, `currentTheta`, and the target angle, `controllerAngle`, both in degrees, and an `LEDClass` object called `ontargetLED`. Complete the `TODO` from the lab in the cell below.


In [ ]:
# Flip LED off/on depending on whether we are in range
onTarget = abs(currentTheta - controllerAngle) < 5    # True when within 5 degrees of the target
if onTarget != ontargetLED.state:                      # the LED is showing the wrong thing
    ontargetLED.toggle()


**Explanation.** `LEDClass` has no `on()` or `off()` function, only `toggle()`. So the code compares what the LED *should* show, `onTarget`, with what it *does* show, `ontargetLED.state`, and toggles only when the two differ. Calling `toggle()` every time through the loop while the pendulum is on target would make the LED flicker instead of staying lit.

The same logic written out with `if` and `else` is also correct:

```python
if abs(currentTheta - controllerAngle) < 5:
    if not ontargetLED.state:      # should be on, but it is off
        ontargetLED.toggle()
else:
    if ontargetLED.state:          # should be off, but it is on
        ontargetLED.toggle()
```

Using `<= 5` instead of `< 5` is fine too.